# 01 Analytical flow model of the focusing channel

This notebook computes the flow quantities of the Results that follow from the models of the Background chapter
and from the estimates described in Methods (`sec:methods_operating`, `sec:methods_flow_deformation`). All
functions are in `flow_theory.py`. The notebook explains each equation and shows the resulting tables.

**Outputs** (`results/flow_theory/`)

| File | Content | Thesis |
|---|---|---|
| `operating_points.csv` | velocities, Re, pressure gradient, pressure drop, wall shear stress | Table `tab:res-flow`, Section `sec:res-flow` |
| `hydraulic_resistance.csv` | exact series against Eq. `hydraulic_resistance_rectangular` | Background, Section `sec:res-flow` |
| `stream_width.csv`, `stream_width_by_ratio.csv` | model of Lee et al. against the measured stream width | Table `tab:res-stream` |
| `cell_loads.csv` | particle Reynolds number, shear stress, capillary number, focusing length, forces | Section `sec:res-loads` |
| `motion_blur.csv` | motion blur of the Phantom recordings | Figure `res_motion_blur_phantom` |
| `checks.txt` | every value of the thesis text that this notebook reproduces | |

**Conventions**

* $U$ is the **mean** velocity in the channel cross-section, $U=Q/A$. The centreline velocity is $U_\mathrm{max}$.
* Flow rates are given in µL/min and converted with $1\ \mu\mathrm{L/min}=10^{-9}/60\ \mathrm{m^3/s}$.

In [ ]:
import pandas as pd
from IPython.display import display

import flow_theory as ft

pd.set_option('display.precision', 4)
pd.set_option('display.width', 200)

## 1 Input parameters

| Symbol | Meaning | Value | Source |
|---|---|---|---|
| $\rho$ | density of the fluid (PBS treated as water) | 1000 kg/m³ | Bruus (2008) |
| $\eta$ | dynamic viscosity of the fluid | 1.0 mPa s | Bruus (2008) |
| $w$, $h$ | width and height of the focusing channel | 40 µm, 40 µm | design values |
| $L$ | length of the focusing channel | 3.0 mm | design value |
| $Q$ | total flow rate | 30, 150, 300 µL/min (600 µL/min: bonding failed) | Methods |
| $Q_\mathrm{i}$ | sample flow rate | $Q/3$ (sample to sheath 1:2) | Methods |
| $Q_\mathrm{s}$ | flow rate of **one** sheath stream | $Q/3$ (equal split assumed) | Methods |
| $d_\mathrm{p}$ | cell diameters for the load estimates | 10, 15, 20, 25 µm | Results |
| $E_\mathrm{app}$ | apparent Young's modulus of the cell | 0.3 to 1.5 kPa | Armistead et al. (2019), Fregin et al. (2019) |
| $f_\mathrm{L}$ | dimensionless lift coefficient | 0.02 to 0.05 | Amini et al. (2014), p. 2743 |
| $p_\mathrm{obj}$ | pixel size in the object plane (Phantom TMX) | 0.925 µm | Methods |
| $t_\mathrm{exp}$ | exposure times drawn in the motion-blur figure | 0.2 and 0.7 µs | recording headers |

The outlet hole was punched at the end of the 3.0 mm focusing channel in all experiments. The pressure drop on the
chip is therefore that of the focusing channel alone.

**Measured inputs:** the stream width per condition (`MEASURED_STREAM`, output of the stream-width image analysis)
and the lowest and highest median cell velocity per flow rate (`MEASURED_CELL_VELOCITY`, from the tracked cells).

In [ ]:
print('rho =', ft.RHO, 'kg/m^3, eta =', ft.ETA, 'Pa s')
print('channel:', ft.W_CH / ft.UM, 'x', ft.H_CH / ft.UM, 'µm, length', ft.L_CH * 1e3, 'mm')
print('flow rates:', ft.Q_TOTAL, 'µL/min, failed at', ft.Q_FAILED, 'µL/min')
display(ft.MEASURED_STREAM)
print('measured median cell velocity (lowest, highest) in m/s:', ft.MEASURED_CELL_VELOCITY)

## 2 Models

### 2.1 Velocity profile in the rectangular channel

Eq. `poiseuille_rectangular_velocity` (Mortensen et al. 2005):

$$
u_x(y,z)=\frac{4h^2\,\Delta p}{\pi^3\eta L}\sum_{n=1,3,5,\dots}^{\infty}\frac{1}{n^3}
\left[1-\frac{\cosh(n\pi y/h)}{\cosh\left(n\pi w/(2h)\right)}\right]\sin\left(\frac{n\pi z}{h}\right)
$$

where $u_x(y,z)$ is the axial velocity, $y$ the lateral coordinate ($-w/2<y<w/2$, $y=0$ on the centreline),
$z$ the coordinate along the depth ($0<z<h$, mid-plane at $z=h/2$), $\Delta p/L$ the pressure gradient and $\eta$
the dynamic viscosity. The series is truncated after 200 odd terms (Methods). Function: `ft.velocity_rect`.

### 2.2 Flow rate, hydraulic resistance and maximum velocity

Integrating the series over the cross-section (Eq. `volume_flow_rate`) gives

$$
Q=\frac{h^3w}{12\eta}\,\frac{\Delta p}{L}\left[1-\frac{192\,h}{\pi^5 w}\sum_{n=1,3,5,\dots}^{\infty}\frac{\tanh\left(n\pi w/(2h)\right)}{n^5}\right],
$$

and with Eq. `hagen_poiseuille_general`, $\Delta p=R_\mathrm{hyd}Q$, the hydraulic resistance $R_\mathrm{hyd}$.
All pressure values use this full series (`ft.r_hyd_exact`). The approximation of Eq.
`hydraulic_resistance_rectangular`,

$$R_\mathrm{hyd}\simeq\frac{12\,\eta L}{h^3w}\left(1-0.630\,\frac{h}{w}\right)^{-1},$$

is valid for $h<w$ and is only shown for comparison (`ft.r_hyd_approx`). For a square channel Bruus (2008) gives
$R_\mathrm{hyd}=28.4\,\eta L/h^4$. The centreline velocity is $U_\mathrm{max}=u_x(0,h/2)$, and the ratio
$U_\mathrm{max}/U$ depends only on the aspect ratio $w/h$.

### 2.3 Shear stress

Shear stress in the mid-plane at the lateral position $y$ (Gerum et al. 2022, Eq. 4; Eq. `shear_stress_rectangular`):

$$
\tau(y)=\frac{4h}{\pi^{2}}\,\frac{\Delta p}{L}\sum_{n=1,3,5,\dots}^{\infty}\frac{(-1)^{(n-1)/2}}{n^{2}}\,
\frac{\sinh\left(n\pi y/h\right)}{\cosh\left(n\pi w/(2h)\right)}.
$$

Its value at $y=w/2$ is the wall shear stress at the centre of the side wall. The mean wall shear stress follows from
the force balance on the fluid (Methods), $\bar\tau_\mathrm{w}=(\Delta p/L)\,wh/[2(w+h)]$. The shear rate is
$\dot\gamma=\tau/\eta$. Function: `ft.shear_midplane`.

### 2.4 Width of the focused stream (Lee et al. 2006)

Eq. `hydrodynamic_focusing_width`:
$$\frac{w_\mathrm{f}}{w_\mathrm{o}}=\frac{Q_\mathrm{i}}{\gamma\left(Q_\mathrm{i}+Q_{\mathrm{s},1}+Q_{\mathrm{s},2}\right)}$$

Eq. `velocity_ratio_gamma`:
$$
\gamma=\frac{1-\dfrac{192h}{\pi^5 w_\mathrm{f}}\displaystyle\sum_{n=0}^{\infty}\frac{1}{(2n+1)^5}\,
\frac{\sinh\left[(2n+1)\pi w_\mathrm{f}/(2h)\right]}{\cosh\left[(2n+1)\pi w_\mathrm{o}/(2h)\right]}}
{1-\dfrac{192h}{\pi^5 w_\mathrm{o}}\displaystyle\sum_{n=0}^{\infty}\frac{\tanh\left[(2n+1)\pi w_\mathrm{o}/(2h)\right]}{(2n+1)^5}}
$$

where $w_\mathrm{f}$ is the width of the focused stream, $w_\mathrm{o}$ the width of the focusing channel,
$Q_\mathrm{i}$ the sample flow rate and $Q_{\mathrm{s},1}=Q_{\mathrm{s},2}$ the flow rates of the two sheath streams.
Because $\gamma$ depends on $w_\mathrm{f}$, both equations are solved together by bisection (`ft.lee_width`).
`ft.lee_gamma_numeric` checks $\gamma$ by direct integration of the depth-averaged profile of Section 2.1.

**Combining conditions.** Conditions $j$ with mean $\bar w_j$, standard deviation $s_j$ and $n_j$ images are combined
over all $N=\sum_j n_j$ images (`ft.pool_groups`):

$$\bar w=\frac{1}{N}\sum_j n_j\bar w_j,\qquad
s=\sqrt{\frac{\sum_j\left[(n_j-1)\,s_j^2+n_j(\bar w_j-\bar w)^2\right]}{N-1}}.$$

## 3 Operating points (Table `tab:res-flow`)

For each total flow rate: $U=Q/(wh)$, $U_\mathrm{max}$, the mean velocity in the focused stream $\gamma U$, the
Reynolds number (Eq. `reynolds_number`) $\mathrm{Re}=\rho U D_\mathrm{h}/\eta$ with $D_\mathrm{h}=2wh/(w+h)=40$ µm
(`Re_max` with $U_\mathrm{max}$), the pressure gradient $\Delta p/L=R_\mathrm{hyd}Q/L$, the pressure drop over the
focusing channel (lower bound, filters, inlets, outlet hole and tubing excluded), the wall shear stress, the
residence time $t_\mathrm{res}=L/U$ and the stream width of Lee et al.

In [ ]:
ops = ft.table_operating_points()
display(ops)
display(ft.table_resistances())

## 4 Stream width (Table `tab:res-stream`)

Per condition, the model is evaluated with the measured channel width of the condition (`lee_um`) and with the design
width of 40 µm (`lee_nominal_40um`). `without_gamma_um` is $w_\mathrm{o}Q_\mathrm{i}/Q$. The second table combines all
images with the same sheath-to-sample ratio $Q_\mathrm{sheath}/Q_\mathrm{i}$ (ratio 2: total flow rates of 3, 30 and
150 µL/min, $N=9$). `lee_measured_um` is the image-weighted mean of the per-condition predictions (column "Lee model"
of Table `tab:res-stream`).

In [ ]:
sw = ft.table_stream_width()
display(sw)
display(ft.table_stream_by_ratio(sw))

## 5 Loads on the cells (Section `sec:res-loads`)

For a cell of diameter $d_\mathrm{p}$ in the undisturbed, fully developed flow:

* confinement ratio, Eq. `confinement-ratio`: $\chi=d_\mathrm{p}/H$ with $H=w=h=40$ µm,
* particle Reynolds number, Eq. `particle_reynolds_number`: $\mathrm{Re}_\mathrm{p}=\mathrm{Re}\,(d_\mathrm{p}/H)^2$
  (`Re_p_max` with $U_\mathrm{max}$),
* shear stress at the edge of a centred cell, $\sigma_\mathrm{c}=\tau(d_\mathrm{p}/2)$, and at the outer edge of a cell
  whose centre lies at the boundary of the focused stream, $\tau(w_\mathrm{f}/2+d_\mathrm{p}/2)$,
* capillary number, Eq. `ca_number`: $\mathrm{Ca}=\eta\dot\gamma/G$ with $\dot\gamma=\sigma_\mathrm{c}/\eta$ and
  $G=E_\mathrm{app}/3$,
* inertial focusing length, Eq. `focusing_length`:
  $L_\mathrm{f}=\pi\eta H^{2}/(\rho\,U_\mathrm{max}\,d_\mathrm{p}^{2}f_\mathrm{L})$,
* inertial lift force, Eq. `inertial_lift_force`: $F_\mathrm{L}=f_\mathrm{L}\,\rho\,U_\mathrm{max}^2d_\mathrm{p}^4/H^2$,
* Stokes drag per relative velocity, Eq. `stokes_drag`: $F_\mathrm{D}/U_\mathrm{rel}=3\pi\eta d_\mathrm{p}$.

In [ ]:
loads = pd.concat([ft.table_loads(q) for q in ft.Q_TOTAL], ignore_index=True)
display(loads[loads.Q_total_ul_min == 300])

## 6 Motion blur of the Phantom recordings

Eq. `motion_blur`: $\Delta x_\mathrm{blur}=v\,t_\mathrm{exp}$, and Eq. `max_exposure_time` for a blur of at most one
pixel: $t_\mathrm{exp}\le p_\mathrm{obj}/v$. The velocity $v$ is the highest measured median cell velocity at each flow
rate.

In [ ]:
display(ft.table_motion_blur())

## 7 Save all tables and compare with the thesis text

`checks.txt` lists every value of the thesis text that follows from this model. The value in the text is given in
brackets.

In [ ]:
res = ft.run(save=True)
print(res['checks'])
print('\nwritten to', ft.RESULTS_DIR.resolve())